In [1]:
import pandas as pd
import numpy as np

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

from sklearn.metrics import mean_absolute_error, mean_squared_error

In [2]:
df = pd.read_csv("../data/processed/store_daily_sales.csv")

df["Date"] = pd.to_datetime(df["Date"])

df = df.sort_values(["Store", "Date"])

store_1 = df[df["Store"] == 1].copy()

store_1 = store_1.sort_values("Date")

sales = store_1["Sales"]

train = sales.iloc[:-30]
test = sales.iloc[-30:]

print("Training:", len(train))
print("Testing :", len(test))

Training: 751
Testing : 30


In [4]:
def smape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    denominator = np.abs(y_true) + np.abs(y_pred)

    return np.mean(
        np.where(
            denominator == 0,
            0,
            2 * np.abs(y_pred - y_true) / denominator
        )
    ) * 100

In [5]:
naive_prediction = np.repeat(
    train.iloc[-1],
    len(test)
)

print(naive_prediction[:5])

[3317 3317 3317 3317 3317]


In [6]:
seasonal_naive_prediction = sales.shift(7).iloc[-30:]

print(seasonal_naive_prediction.head())

751    4202.0
752    4097.0
753    3846.0
754    3762.0
755    3346.0
Name: Sales, dtype: float64


In [7]:
arima_model = ARIMA(
    train,
    order=(1, 0, 1)
).fit()

arima_prediction = arima_model.forecast(
    steps=30
)

print(arima_prediction.head())

751    3768.982585
752    4066.064384
753    4274.786354
754    4421.428997
755    4524.456317
Name: predicted_mean, dtype: float64


In [8]:
train_values = pd.Series(train.values)

sarima_model = SARIMAX(
    train_values,
    order=(1, 0, 1),
    seasonal_order=(1, 0, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

sarima_prediction = sarima_model.forecast(
    steps=30
)

print(sarima_prediction[:5])

751    3393.510472
752    3385.852405
753    3395.304695
754    3367.583139
755    3423.487260
Name: predicted_mean, dtype: float64


In [9]:
def evaluate_model(model_name, actual, prediction):
    
    return {
        "Model": model_name,
        "MAE": mean_absolute_error(actual, prediction),
        "RMSE": mean_squared_error(actual, prediction) ** 0.5,
        "sMAPE": smape(actual, prediction)
    }

In [10]:
results = pd.DataFrame([
    evaluate_model(
        "Naive",
        test,
        naive_prediction
    ),
    
    evaluate_model(
        "Seasonal Naive",
        test,
        seasonal_naive_prediction
    ),
    
    evaluate_model(
        "ARIMA(1,0,1)",
        test,
        arima_prediction
    ),
    
    evaluate_model(
        "SARIMA(1,0,1)(1,0,1,7)",
        test,
        sarima_prediction
    )
])

results = results.sort_values("MAE")

results.round(2)

,Model,MAE,RMSE,sMAPE
2,"ARIMA(1,0,1)",642.08,786.10,14.23
1,Seasonal Naive,1148.83,1261.86,25.99
0,Naive,1223.57,1409.59,29.80
3,"SARIMA(1,0,1)(1,0,1,7)",1328.75,1508.05,33.01


In [11]:
best_model = results.iloc[0]

print("Best model:", best_model["Model"])
print("MAE:", round(best_model["MAE"], 2))
print("RMSE:", round(best_model["RMSE"], 2))
print("sMAPE:", round(best_model["sMAPE"], 2), "%")

Best model: ARIMA(1,0,1)
MAE: 642.08
RMSE: 786.1
sMAPE: 14.23 %


In [12]:
seasonal_naive_mae = results.loc[
    results["Model"] == "Seasonal Naive",
    "MAE"
].iloc[0]

best_mae = best_model["MAE"]

improvement = (
    (seasonal_naive_mae - best_mae)
    / seasonal_naive_mae
) * 100

print(
    "Improvement vs Seasonal Naive:",
    round(improvement, 2),
    "%"
)

Improvement vs Seasonal Naive: 44.11 %


In [13]:
results.to_csv(
    "../data/processed/day7_model_evaluation.csv",
    index=False
)

print("Evaluation results saved successfully!")

Evaluation results saved successfully!


# Day 7 Findings

## Evaluation Setup

Models were evaluated on the same Store 1 time-based holdout consisting of the final 30 days.

The following models were compared:

- Naive
- Seasonal Naive
- ARIMA(1,0,1)
- SARIMA(1,0,1)(1,0,1,7)

Evaluation metrics:

- MAE
- RMSE
- sMAPE

## Conclusion

The models were compared using identical validation dates and forecast horizons.

The model with the lowest validation error was selected as the best-performing model for this evaluation window.

These results are specific to the selected Store 1 holdout and should not be generalized to all stores without further validation.